###1) Konfiguracja

In [0]:
from functools import reduce
from datetime import datetime, timezone
import uuid

from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

RUN_ID = str(uuid.uuid4())

DATA_PATH = (
    "/Volumes/workspace/energy_bronze/"
    "landing/nlr/system_2107/data"
)

FILES = {
    "meter": [
        f"{DATA_PATH}/2107_meter_15m_data.csv",
        f"{DATA_PATH}/2107_meter_15m_data_2024.csv",
    ],
    "environment": [
        f"{DATA_PATH}/2107_environment_data.csv",
        f"{DATA_PATH}/2107_environment_data_2024.csv",
    ],
    "irradiance": [
        f"{DATA_PATH}/2107_irradiance_data.csv",
        f"{DATA_PATH}/2107_irradiance_data_2024.csv",
    ],
}

print(f"Bronze run ID: {RUN_ID}")

###2) Funkcja odczytu CSV

In [0]:
def read_bronze_csv(dataset_name, paths):
    dataframes = []

    for path in paths:
        dataframe = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "false")
            .option("mode", "PERMISSIVE")
            .option("encoding", "UTF-8")
            .load(path)
            .select(
                "*",
                F.col("_metadata.file_path").alias("_source_file_path"),
                F.col("_metadata.file_name").alias("_source_file_name"),
                F.col("_metadata.file_size").alias("_source_file_size")
            )
        )

        dataframes.append(dataframe)

    combined = reduce(
        lambda left, right: left.unionByName(
            right,
            allowMissingColumns=True
        ),
        dataframes
    )

    raw_columns = [
        column
        for column in combined.columns
        if not column.startswith("_")
    ]

    record_hash = F.sha2(
        F.concat_ws(
            "||",
            *[
                F.coalesce(
                    F.col(column).cast("string"),
                    F.lit("")
                )
                for column in sorted(raw_columns)
            ]
        ),
        256
    )

    return (
        combined
        .withColumn("_dataset_name", F.lit(dataset_name))
        .withColumn("_source_system", F.lit("NLR_PVDAQ"))
        .withColumn("_system_id", F.lit(2107))
        .withColumn("_ingestion_run_id", F.lit(RUN_ID))
        .withColumn("_ingested_at_utc", F.current_timestamp())
        .withColumn("_record_hash", record_hash)
    )

###3) Utworzenie DataFrame

In [0]:
meter_bronze = read_bronze_csv(
    "meter",
    FILES["meter"]
)

environment_bronze = read_bronze_csv(
    "environment",
    FILES["environment"]
)

irradiance_bronze = read_bronze_csv(
    "irradiance",
    FILES["irradiance"]
)

print("METER:")
print(meter_bronze.columns)

print("\nENVIRONMENT:")
print(environment_bronze.columns)

print("\nIRRADIANCE:")
print(irradiance_bronze.columns)

###4) Podgląd danych

In [0]:
display(meter_bronze.limit(10))

In [0]:
display(environment_bronze.limit(10))

In [0]:
display(irradiance_bronze.limit(10))

###5) Zapis tabel Delta

In [0]:
BRONZE_TABLES = {
    "meter": (
        meter_bronze,
        "workspace.energy_bronze.nlr_meter_raw"
    ),
    "environment": (
        environment_bronze,
        "workspace.energy_bronze.nlr_environment_raw"
    ),
    "irradiance": (
        irradiance_bronze,
        "workspace.energy_bronze.nlr_irradiance_raw"
    ),
}

for dataset_name, (dataframe, table_name) in BRONZE_TABLES.items():
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    print(f"Zapisano: {table_name}")

###6) Profil danych

In [0]:
profile_rows = []

for dataset_name, (dataframe, table_name) in BRONZE_TABLES.items():
    result = (
        dataframe
        .agg(
            F.count("*").alias("row_count"),
            F.countDistinct("_record_hash").alias(
                "distinct_record_count"
            ),
            F.min("measured_on").alias("min_measured_on_raw"),
            F.max("measured_on").alias("max_measured_on_raw")
        )
        .first()
    )

    profile_rows.append({
        "dataset": dataset_name,
        "table_name": table_name,
        "business_columns": len([
            column
            for column in dataframe.columns
            if not column.startswith("_")
        ]),
        "row_count": result["row_count"],
        "distinct_record_count": result["distinct_record_count"],
        "duplicate_record_count": (
            result["row_count"]
            - result["distinct_record_count"]
        ),
        "min_measured_on_raw": result["min_measured_on_raw"],
        "max_measured_on_raw": result["max_measured_on_raw"],
    })

profile_df = spark.createDataFrame(profile_rows)

display(profile_df)

###7) Kontrola tabel

In [0]:
%sql
SHOW TABLES IN workspace.energy_bronze;